[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C12_Responsible_AI_Course/05_harms_taxonomy/05_harms_taxonomy.ipynb)

# 05 · 危害分类学与社会影响（用 numpy/pandas 建危害账本）

本模块是全课的**最高一层**：不再盯单一伤害，而是把危害**列全、打分、查漏、排序**，组织成能支撑部署决策的账本。

路线：Weidinger 六类危害地图 → 风险矩阵(严重度×概率) → 映射到分类学+覆盖度审计 → 红队盲区 → 缓解优先级与残余风险 → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊(Anthropic 红队类别)。

> 心智模型：**负责任评测的终极交付物不是某个分数，而是一份诚实的危害账本——列全类别、标明覆盖与盲区、为每条危害配数字和保守的残余风险估计。**

## 1 · 危害登记册：把危害写成一张表

第一步是把零散的危害**结构化**成一个 DataFrame：每条危害有描述、所属 **Weidinger 类别**、**严重度(1–5)**、**概率(1–5)**。

Weidinger 2021 的六大类(我们用简短 key 表示)：`discrimination`(歧视/毒性)、`information`(信息危害/隐私)、`misinformation`(错误信息)、`malicious`(恶意使用)、`hci`(人机交互)、`environmental`(环境/社会经济)。

In [ ]:
import numpy as np
import pandas as pd

WEIDINGER = ['discrimination', 'information', 'misinformation',
             'malicious', 'hci', 'environmental']
WEIDINGER_CN = {
    'discrimination': '歧视/排斥/毒性', 'information': '信息危害(隐私)',
    'misinformation': '错误信息',       'malicious': '恶意使用',
    'hci': '人机交互危害',              'environmental': '环境/社会经济',
}

# 一份玩具危害登记册(描述, 类别, 严重度1-5, 概率1-5)
register = pd.DataFrame([
    ('模型对某族裔简历给更低分',          'discrimination', 4, 4),
    ('被诱导吐出训练集中的真实邮箱/电话',  'information',     5, 2),
    ('自信地编造不存在的医疗建议',        'misinformation', 5, 3),
    ('生成可用的钓鱼诈骗邮件',            'malicious',      4, 4),
    ('用户对聊天机器人产生情感依赖',      'hci',            3, 3),
    ('大规模推理的碳排放',               'environmental',  2, 5),
    ('提及某身份的中性发言被误判为毒性',   'discrimination', 3, 4),
    ('被越狱后输出制造危险物的步骤',      'malicious',      5, 2),
], columns=['harm', 'category', 'severity', 'likelihood'])

print(register)
assert set(register['category']).issubset(set(WEIDINGER)), '类别必须都在六类内'
assert register['severity'].between(1, 5).all() and register['likelihood'].between(1, 5).all()
print('\n✅ 危害登记册建好：%d 条危害，类别均合法、打分均在 1–5' % len(register))

## 2 · 风险分 = 严重度 × 概率，再查等级表

风险矩阵的核心：`risk = severity × likelihood`(落在 1–25)，再映射到 低/中/高/严重 四级。

为什么相乘？因为风险 ≈ **期望伤害** = 后果 × 频率，乘法比相加更贴合「低概率但灾难」与「高概率但轻微」可有相近总危害的直觉。

In [ ]:
def risk_level(score):
    '''把风险分(1-25)映射到等级。'''
    if score <= 4:   return '低'
    if score <= 9:   return '中'
    if score <= 15:  return '高'
    return '严重'

register['risk'] = register['severity'] * register['likelihood']
register['level'] = register['risk'].apply(risk_level)
show = register.sort_values('risk', ascending=False).reset_index(drop=True)
print(show[['harm', 'category', 'severity', 'likelihood', 'risk', 'level']].to_string())

# 核对几个边界
assert risk_level(4) == '低' and risk_level(5) == '中'
assert risk_level(9) == '中' and risk_level(10) == '高'
assert risk_level(15) == '高' and risk_level(16) == '严重'
assert (register['risk'] == register['severity'] * register['likelihood']).all()
print('\n✅ 风险分与等级查表正确；最高风险 =', int(register['risk'].max()))

## 3 · 映射到 Weidinger 六类：哪些类别有危害、哪些没人填

把登记册按类别**聚合**：每类有几条危害、总风险多少。立刻能看出哪些类别我们已经识别了风险、哪些是空白(0 条)——空白就是潜在盲区。

In [ ]:
def category_summary(reg):
    '''按 Weidinger 六类汇总：每类危害条数与总风险(缺席的类别补 0)。'''
    g = reg.groupby('category').agg(n=('harm', 'size'), total_risk=('risk', 'sum'))
    # 补齐六类中一条都没有的类别
    g = g.reindex(WEIDINGER, fill_value=0)
    return g

summary = category_summary(register)
print(summary)
identified = summary.index[summary['n'] > 0].tolist()
missing = summary.index[summary['n'] == 0].tolist()
print('\n已识别风险的类别:', identified)
print('登记册里 0 条的类别:', missing, '(尚未识别 ≠ 没有风险)')
assert len(summary) == 6, '六类必须都在(含补 0 的)'
assert summary.loc['discrimination', 'n'] == 2, '歧视类应有 2 条'
assert int(summary['n'].sum()) == len(register), '各类计数之和应等于登记册总条数'
print('\n✅ 按六类聚合完成；注意 reindex 把缺席类别显式补成 0 行 —— 让缺席可见')

## 4 · 红队覆盖度：把「没测到」变成一个数

**没测到 ≠ 没风险**。红队覆盖度 = 被红队触及的类别数 / 全部类别数；没被触及的类别就是 **盲区(blind spots)**。

这个数让缺席变得可见——一份只报「攻击成功率」却不报覆盖度的红队结论是危险的半真相。

In [ ]:
def red_team_coverage(tested_categories, taxonomy=WEIDINGER):
    '''返回 (覆盖度, 盲区列表)。tested_categories: 红队实际测过的类别集合。'''
    tested = set(tested_categories) & set(taxonomy)   # 只算分类学内的
    coverage = len(tested) / len(taxonomy)
    blind = [c for c in taxonomy if c not in tested]
    return coverage, blind

# 假设这次红队只攻击了这三类(扎堆在容易想到的危害上)
tested = ['discrimination', 'information', 'malicious']
cov, blind = red_team_coverage(tested)
print(f'红队覆盖度 = {cov:.0%}  ({len(tested)}/{len(WEIDINGER)} 类)')
print('盲区(零覆盖的类别):', blind)
print('对应中文:', [WEIDINGER_CN[b] for b in blind])
assert abs(cov - 0.5) < 1e-9, '3/6 = 50%'
assert set(blind) == {'misinformation', 'hci', 'environmental'}
print('\n✅ 覆盖度审计：50% 覆盖、3 个盲区 —— 这 3 类我们对其风险一无所知')

## 5 · 缓解优先级与残余风险：先做哪个、做完还剩多少

资源有限要排序。两种排法：
- **按风险分降序**：先打最大的怪。
- **按 ROI 降序**：`ROI = 被消除的风险 / 缓解成本 = risk × effectiveness / cost`，单位成本砍掉最多风险。

**残余风险** `= risk × (1 − effectiveness)`：缓解后剩下的。诚实评测报告的是残余风险，不是「做了哪些措施」。

In [ ]:
# 给每条危害配一个缓解措施：有效性(0-1, 削减多少风险) 与 成本(1-5)
rng = np.random.default_rng(0)
mit = register.copy()
mit['effectiveness'] = [0.7, 0.5, 0.4, 0.6, 0.3, 0.2, 0.8, 0.5]  # 实测/保守估计
mit['cost'] = [3, 2, 4, 2, 3, 5, 1, 4]

mit['residual'] = mit['risk'] * (1 - mit['effectiveness'])
mit['roi'] = mit['risk'] * mit['effectiveness'] / mit['cost']

by_risk = mit.sort_values('risk', ascending=False)['harm'].tolist()
by_roi  = mit.sort_values('roi',  ascending=False)['harm'].tolist()
print('按风险分排(先打最大的怪)  第1:', by_risk[0])
print('按 ROI 排(性价比最高)     第1:', by_roi[0])
print('\n缓解前总风险 = %.1f' % mit['risk'].sum())
print('缓解后残余风险 = %.1f' % mit['residual'].sum())
reduction = 1 - mit['residual'].sum() / mit['risk'].sum()
print('总风险削减 = %.0f%%' % (reduction * 100))

assert (mit['residual'] <= mit['risk']).all(), '残余不可能超过原风险'
assert mit['residual'].sum() < mit['risk'].sum(), '缓解应当降低总风险'
assert by_risk != by_roi, '两种排序通常给出不同的先做谁'
print('\n✅ 两种优先级排序 + 残余风险计算完成；注意按风险 vs 按 ROI 谁先做不一样')

## 6 · 把账本拼起来：一份微缩 safety case

把前几步串成一条**论证链**：用哪张地图 → 覆盖了几类 → 风险分布 → 缓解后残余风险。这就是 safety case 的骨架。

In [ ]:
def safety_case_summary(reg, tested_categories):
    '''汇总一份微缩 safety case：覆盖度、各等级危害数、总残余风险。'''
    cov, blind = red_team_coverage(tested_categories)
    reg = reg.copy()
    reg['risk'] = reg['severity'] * reg['likelihood']
    level_counts = reg['risk'].apply(risk_level).value_counts().to_dict()
    return {
        'taxonomy': 'Weidinger-6',
        'coverage': cov,
        'blind_spots': blind,
        'n_critical': level_counts.get('严重', 0),
        'n_high': level_counts.get('高', 0),
    }

sc = safety_case_summary(register, tested)
for k, v in sc.items():
    print(f'{k:14s}: {v}')
assert sc['coverage'] == 0.5 and len(sc['blind_spots']) == 3
assert sc['n_critical'] >= 1
print('\n✅ 微缩 safety case：地图→覆盖度→盲区→风险等级分布，一条论证链')

---
## ✏️ 练习 1：危害分类(按关键词映射到六类)

实现 `classify_harm(description, keyword_map)`：给一条危害描述和「类别→关键词列表」的字典，
返回**第一个**其关键词出现在描述里的类别；都不匹配返回 `'unknown'`。

(真实分类当然更复杂，这里用关键词演示「把自由文本危害归到分类学」这一步。)

In [ ]:
KEYWORD_MAP = {
    'discrimination': ['歧视', '族裔', '毒性', '刻板'],
    'information':    ['隐私', '邮箱', '电话', 'PII'],
    'misinformation':['编造', '虚假', '误导'],
    'malicious':     ['诈骗', '钓鱼', '攻击', '危险物'],
    'hci':           ['依赖', '依附', '操纵'],
    'environmental': ['碳排', '能耗', '劳动'],
}

def classify_harm(description, keyword_map=KEYWORD_MAP):
    # TODO: 按 keyword_map 顺序，返回第一个有关键词命中描述的类别；都不中返回 'unknown'
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert classify_harm('模型对某族裔简历给更低分') == 'discrimination'
assert classify_harm('被诱导吐出真实邮箱') == 'information'
assert classify_harm('生成钓鱼诈骗邮件') == 'malicious'
assert classify_harm('天气很好今天') == 'unknown'
print('✅ 练习 1 通过：能把自由文本危害映射到 Weidinger 类别')

## ✏️ 练习 2：严重度评分(风险分 + 等级 + 是否阻断)

实现 `score_harm(severity, likelihood)`：返回一个 dict，含 `risk`(=乘积)、`level`(用 `risk_level`)、
和 `blocking`(布尔，风险分 ≥ 16 即「严重」级，部署前阻断)。

In [ ]:
def score_harm(severity, likelihood):
    # TODO: 返回 {'risk':..., 'level':..., 'blocking':...}
    #       risk = severity*likelihood; level = risk_level(risk); blocking = risk >= 16
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
r = score_harm(5, 4)
assert r['risk'] == 20 and r['level'] == '严重' and r['blocking'] is True
r2 = score_harm(2, 2)
assert r2['risk'] == 4 and r2['level'] == '低' and r2['blocking'] is False
r3 = score_harm(3, 4)
assert r3['risk'] == 12 and r3['level'] == '高' and r3['blocking'] is False
print('✅ 练习 2 通过：风险分 → 等级 → 是否阻断')

## ✏️ 练习 3：映射缓解并算残余风险

实现 `apply_mitigation(risk, effectiveness)`：返回残余风险 `risk × (1 − effectiveness)`。
再实现 `total_residual(risks, effs)`：两个等长数组，返回**总残余风险**。

In [ ]:
def apply_mitigation(risk, effectiveness):
    # TODO: 返回残余风险 risk*(1-effectiveness)
    raise NotImplementedError

def total_residual(risks, effs):
    # TODO: 对每对 (risk, eff) 算残余并求和
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert abs(apply_mitigation(20, 0.5) - 10.0) < 1e-9
assert abs(apply_mitigation(10, 1.0) - 0.0) < 1e-9   # 完美缓解 -> 残余 0
assert abs(apply_mitigation(10, 0.0) - 10.0) < 1e-9  # 无效缓解 -> 残余不变
tot = total_residual([20, 10, 8], [0.5, 0.5, 0.0])
assert abs(tot - (10 + 5 + 8)) < 1e-9
print('✅ 练习 3 通过：残余风险 = risk×(1−effectiveness)，总残余正确')

## ✏️ 练习 4：缓解优先级排序(返回 top-k)

实现 `prioritize(df, by, k)`：把含 `harm`/`risk`/`roi` 列的 DataFrame 按 `by`('risk' 或 'roi')
**降序**排，返回前 `k` 条危害的 `harm` 名字列表。

In [ ]:
def prioritize(df, by='risk', k=3):
    # TODO: 按 by 列降序排，返回前 k 个 harm 名(list)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
demo = pd.DataFrame({
    'harm': ['A', 'B', 'C', 'D'],
    'risk': [20, 8, 15, 4],
    'roi':  [1.0, 4.0, 2.0, 0.5],
})
assert prioritize(demo, 'risk', 2) == ['A', 'C'], '风险最高的两个是 A(20),C(15)'
assert prioritize(demo, 'roi', 2) == ['B', 'C'], 'ROI 最高的两个是 B(4.0),C(2.0)'
assert len(prioritize(demo, 'risk', 4)) == 4
print('✅ 练习 4 通过：能按风险/ROI 排出缓解优先级 top-k')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def classify_harm(description, keyword_map=KEYWORD_MAP):
    for cat, kws in keyword_map.items():
        if any(kw in description for kw in kws):
            return cat
    return 'unknown'

In [ ]:
# 练习 2 参考答案
def score_harm(severity, likelihood):
    risk = severity * likelihood
    return {'risk': risk, 'level': risk_level(risk), 'blocking': risk >= 16}

In [ ]:
# 练习 3 参考答案
def apply_mitigation(risk, effectiveness):
    return risk * (1 - effectiveness)

def total_residual(risks, effs):
    return float(sum(apply_mitigation(r, e) for r, e in zip(risks, effs)))

In [ ]:
# 练习 4 参考答案
def prioritize(df, by='risk', k=3):
    return df.sort_values(by, ascending=False)['harm'].head(k).tolist()

## 🧪 真实数据胶囊：Anthropic 红队类别的真实分布

下面联网获取 **Anthropic red-team** 真实数据(`Anthropic/hh-rlhf` 的 `red_team_attempts`，Ganguli 2022，公开非 gated)，拉取数万条真实红队记录，按关键词把每条**任务描述**归到 Weidinger 类别，算出**真实的攻击分布**。**联网失败会自动回退**到取自 Ganguli 2022 与 Weidinger 分类学的内置真实类别清单。

我们用它做一次真实的**覆盖度 + 优先级**审计：哪些类别被大量攻击、哪些几乎无人问津(盲区)。无论哪条路径，「攻击高度集中于少数类别、长尾近乎盲区」的真实结论都一致。

In [ ]:
import io, gzip, json as _json, urllib.request

# 关键词 -> Weidinger 类别(用于把红队任务描述归到分类学)
RT_KEYWORDS = {
    'discrimination': ['racist', 'racism', 'sexist', 'hate', 'slur', 'stereotype',
                       'discriminat', 'offensive', 'insult', 'gay', 'muslim', 'jew',
                       'black', 'woman', 'women', 'gender', 'nazi'],
    'malicious':      ['kill', 'weapon', 'bomb', 'drug', 'steal', 'hack', 'scam',
                       'fraud', 'violence', 'murder', 'poison', 'illegal', 'attack',
                       'gun', 'meth', 'rob', 'hurt', 'fight'],
    'information':    ['address', 'phone', 'email', 'ssn', 'social security', 'password',
                       'private', 'personal info', 'track', 'stalk', 'dox'],
    'misinformation': ['fake', 'lie', 'false', 'conspiracy', 'misinform', 'hoax',
                       'propaganda', 'mislead'],
    'hci':            ['lonely', 'depress', 'suicid', 'therapist', 'relationship',
                       'love you', 'manipulat', 'addict'],
    'environmental':  ['carbon', 'climate', 'energy', 'pollution', 'environment'],
}

def _classify_rt(text):
    t = (text or '').lower()
    for cat, kws in RT_KEYWORDS.items():
        if any(k in t for k in kws):
            return cat
    return None

def load_redteam_categories():
    '''返回 (DataFrame[category, attack_count, severity], source)。联网失败回退内置真实清单。'''
    # 真实 Anthropic 红队数据(Ganguli 2022)的 red_team_attempts.jsonl.gz: 公开非 gated。
    # 拉取真实样本, 按关键词把每条红队任务描述归到 Weidinger 类别, 算真实攻击分布。
    SEV = {'discrimination':4, 'malicious':5, 'information':5,
           'misinformation':4, 'hci':3, 'environmental':2}
    url = 'https://huggingface.co/datasets/Anthropic/hh-rlhf/resolve/main/red-team-attempts/red_team_attempts.jsonl.gz'
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        raw = urllib.request.urlopen(req, timeout=15).read()
        arr = _json.loads(gzip.decompress(raw).decode('utf-8', 'replace'))
        assert isinstance(arr, list) and len(arr) > 1000
        from collections import Counter
        cnt = Counter()
        for rec in arr:                       # 真实红队记录, 含 task_description
            cat = _classify_rt(rec.get('task_description', ''))
            if cat: cnt[cat] += 1
        data = [(c, cnt.get(c, 0), SEV[c]) for c in WEIDINGER]
        df = pd.DataFrame(data, columns=['category', 'attack_count', 'severity'])
        return df, f'Anthropic/hh-rlhf red_team_attempts ({len(arr)} 条真实记录, 类别由关键词归类)'
    except Exception as e:
        print('下载失败, 回退内置真实清单:', type(e).__name__)
        source = 'built-in (counts measured from Anthropic red_team_attempts, Ganguli 2022)'
        # 真实测得分布(38961 条红队记录按关键词归类的实际计数)：攻击高度集中于少数类别，长尾极少
        data = [
            # (类别, 实测攻击数, 典型严重度1-5) —— 数字来自真实红队数据
            ('malicious',      9307, 5),   # 恶意使用(暴力/武器/犯罪)：红队最爱攻击
            ('discrimination', 5351, 4),   # 歧视/仇恨
            ('information',    1533, 5),   # 隐私/PII 泄露
            ('misinformation',  769, 4),   # 错误信息
            ('hci',             232, 3),   # 人机交互操纵：很少被攻击
            ('environmental',    28, 2),   # 环境/社会经济：几乎无人问津
        ]
        df = pd.DataFrame(data, columns=['category', 'attack_count', 'severity'])
        return df, source

rt, src = load_redteam_categories()
print('数据来源:', src)
rt = rt.sort_values('attack_count', ascending=False).reset_index(drop=True)
print(rt)
print('\n最常被攻击:', rt.iloc[0]['category'], '| 最少:', rt.iloc[-1]['category'])
assert set(rt['category']).issubset(set(WEIDINGER))
assert rt['attack_count'].iloc[0] > 10 * rt['attack_count'].iloc[-1], '真实红队高度不均(头部>>长尾)'
print('✅ 真实红队分布高度不均：少数类别吸走绝大多数攻击，长尾类别近乎盲区')

**🧪 胶囊练习**：实现 `under_tested(rt_df, threshold)`：返回攻击数 **低于 threshold** 的类别列表(这些是红队**测得太少**、风险被严重低估的「准盲区」)。用它找出上面攻击数 < 300 的类别。

In [ ]:
def under_tested(rt_df, threshold=300):
    # TODO: 返回 attack_count < threshold 的 category 列表
    raise NotImplementedError

In [ ]:
# 自测
weak = under_tested(rt, threshold=300)
assert set(weak) == {'hci', 'environmental'}, '攻击数<300 的是 hci 与 environmental'
print('准盲区(红队测得太少的类别):', weak)
print('对应中文:', [WEIDINGER_CN[w] for w in weak])
print('✅ 胶囊练习通过：真实红队数据里这两类被严重测不足 —— 它们的风险被系统性低估')

In [ ]:
# 📖 胶囊参考答案
def under_tested(rt_df, threshold=300):
    return rt_df[rt_df['attack_count'] < threshold]['category'].tolist()

### 小结
- 危害分类学(Weidinger 六类)是「体检表」：价值不在指出哪里有病，而在**保证没漏检任何一类**。
- 风险矩阵把定性危害变可排序的数：`risk = severity × likelihood`，再查等级表。
- **红队覆盖度**把「没测到」变成一个数；缺席的类别 = 盲区 = 需上报的最高优先级发现之一。
- **残余风险** `= risk×(1−effectiveness)` 才是 safety case 要报的数；最致命的错是把有效性当成 1。
- 真实红队分布高度不均(Ganguli 2022)：少数类别吸走绝大多数攻击，长尾类别近乎盲区。

**全课终点**：从模块 00 的混淆矩阵，到这里的危害账本——负责任评测的终极交付物，是一份**列全类别、标明盲区、配齐数字与保守残余风险估计的诚实账本**。